# 03_train_mlp.ipynb

**目标**: 跑通最小 baseline（ResNet18 + MLP），在 LIBERO 单个任务上实现
完整流程。

不追求成功率，纯 sanity check：损失能下降，说明 pipeline 通了。

In [1]:
# 导入库
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm
import importlib.util

# 动态导入自己的 dataset 模块（避免命名问题）
dataset_module_path = Path.cwd() / '02_dataset.py'
spec = importlib.util.spec_from_file_location('dataset02', dataset_module_path)
dataset_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dataset_module)
LIBERODataset = dataset_module.LIBERODataset
create_dataloader = dataset_module.create_dataloader

# 导入 LIBERO 工具
from libero.libero import get_libero_path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.3.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
GPU memory: 8.6 GB


## 1. 数据准备

In [2]:
# 路径配置
dataset_dir = Path(get_libero_path("datasets")) / "libero_spatial"
demo_files = sorted(dataset_dir.glob("*.hdf5"))

print(f"Found {len(demo_files)} demo files:")
for f in demo_files[:3]:
    print(f"  - {f.name}")

# 选择第一个任务
task_file = str(demo_files[0])
print(f"\nUsing task: {Path(task_file).name}")

Found 10 demo files:
  - pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate_demo.hdf5
  - pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_demo.hdf5
  - pick_up_the_black_bowl_in_the_top_drawer_of_the_wooden_cabinet_and_place_it_on_the_plate_demo.hdf5

Using task: pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate_demo.hdf5


In [3]:
# 创建 dataset（单帧采样）
# 后续可改成 seq_len=5 用于 Diffusion Policy
dataset = LIBERODataset(
    task_file,
    seq_len=1,
    image_key="agentview_rgb",
    use_state=False,  # 暂不用 state，只用图像
)

print(f"\nDataset size: {len(dataset)}")

# 随机分割成训练/测试
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

Loaded 50 demos from pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate_demo.hdf5
  Total valid samples (seq_len=1): 5068

Dataset size: 5068
Train size: 4054, Test size: 1014


In [4]:
# 创建 DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

# 检查一个 batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  images shape: {sample_batch['images'].shape}")  # (batch, 3, 128, 128)
print(f"  actions shape: {sample_batch['actions'].shape}")  # (batch, 7)
print(f"  images dtype: {sample_batch['images'].dtype}, range: [{sample_batch['images'].min():.3f}, {sample_batch['images'].max():.3f}]")
print(f"  actions dtype: {sample_batch['actions'].dtype}")
print(f"  actions sample:\n{sample_batch['actions'][0]}")

Train batches: 127, Test batches: 32

Sample batch:
  images shape: torch.Size([32, 3, 128, 128])
  actions shape: torch.Size([32, 7])
  images dtype: torch.float32, range: [0.000, 1.000]
  actions dtype: torch.float32
  actions sample:
tensor([-0.0295,  0.0000, -0.9375,  0.0664, -0.1275, -0.0000,  1.0000])


## 2. 模型定义

In [5]:
from torchvision.models import resnet18

class ImitationLearningModel(nn.Module):
    """ResNet18 + MLP for action prediction."""
    
    def __init__(self, action_dim=7, hidden_dim=256, freeze_backbone=False):
        super().__init__()
        
        # ResNet18 图像编码器
        self.backbone = resnet18(pretrained=False)
        # 移除最后的分类层
        self.backbone.fc = nn.Identity()
        backbone_out_dim = 512
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        # MLP 头：512 -> hidden -> action_dim
        self.head = nn.Sequential(
            nn.Linear(backbone_out_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, action_dim),
        )
    
    def forward(self, images):
        """Forward pass.
        
        Args:
            images: (batch, 3, 128, 128)
        
        Returns:
            actions: (batch, 7)
        """
        features = self.backbone(images)  # (batch, 512)
        actions = self.head(features)      # (batch, 7)
        return actions


# 初始化模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ImitationLearningModel(action_dim=7, hidden_dim=256)
model.to(device)

# 统计参数
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params / 1e6:.1f}M (trainable: {trainable_params / 1e6:.1f}M)")

print(f"\nModel on device: {device}")

/home/dongniuniu/miniconda3/envs/libero/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dongniuniu/miniconda3/envs/libero/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model parameters: 11.4M (trainable: 11.4M)

Model on device: cuda


## 3. 训练配置

In [6]:
# 优化器和损失函数
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# 训练超参
num_epochs = 50
eval_every = 5

# 记录损失
history = {
    'train_loss': [],
    'test_loss': [],
    'lr': [],
}

print(f"Training setup:")
print(f"  Optimizer: Adam (lr=1e-3)")
print(f"  Loss: MSE")
print(f"  Epochs: {num_epochs}")
print(f"  Batch size: {batch_size}")

Training setup:
  Optimizer: Adam (lr=1e-3)
  Loss: MSE
  Epochs: 50
  Batch size: 32


## 4. 训练循环

In [7]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train one epoch."""
    model.train()
    total_loss = 0.0
    
    for batch in tqdm(loader, desc="Train", leave=False):
        images = batch['images'].to(device)
        actions = batch['actions'].to(device)
        
        optimizer.zero_grad()
        pred_actions = model(images)
        loss = criterion(pred_actions, actions)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.shape[0]
    
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, criterion, device):
    """Evaluate on test set."""
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Test", leave=False):
            images = batch['images'].to(device)
            actions = batch['actions'].to(device)
            pred_actions = model(images)
            loss = criterion(pred_actions, actions)
            total_loss += loss.item() * images.shape[0]
    
    return total_loss / len(loader.dataset)


print("Starting training...\n")

for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['lr'].append(scheduler.get_last_lr()[0])
    
    if (epoch + 1) % eval_every == 0:
        test_loss = eval_epoch(model, test_loader, criterion, device)
        history['test_loss'].append(test_loss)
        print(f"Epoch {epoch+1:3d}/{num_epochs} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")
    else:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | Train Loss: {train_loss:.6f}")

print("\nTraining done!")

Starting training...



Epoch   1/50 | Train Loss: 0.113078


Epoch   2/50 | Train Loss: 0.067027


Epoch   3/50 | Train Loss: 0.060793


Epoch   4/50 | Train Loss: 0.054497


...

Epoch  46/50 | Train Loss: 0.003208


Epoch  47/50 | Train Loss: 0.003094


Epoch  48/50 | Train Loss: 0.003034


Epoch  49/50 | Train Loss: 0.002974


Epoch  50/50 | Train Loss: 0.002963 | Test Loss: 0.028925

Training done!


## 5. 结果分析与可视化

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from pathlib import Path

# 如果 history 不在内存里（比如 kernel 重启后），从保存的模型文件加载
if 'history' not in dir() or not history['train_loss']:
    ckpt = torch.load('outputs/model_mlp_baseline.pt', map_location='cpu')
    history = ckpt['history']
    eval_every = 5  # 和训练时保持一致
    print("Loaded history from checkpoint.")
else:
    print("Using history from memory.")

# 绘制损失曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train Loss')
test_epochs = [i * eval_every for i in range(len(history['test_loss']))]
axes[0].plot(test_epochs, history['test_loss'], 'o-', label='Test Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['lr'], label='Learning Rate')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('LR')
axes[1].set_title('Learning Rate Schedule')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
Path('outputs').mkdir(exist_ok=True)
plt.savefig('outputs/training_curves.png', dpi=100, bbox_inches='tight')
plt.close()

print("Saved training curves to outputs/training_curves.png")

Loaded history from checkpoint.
Saved training curves to outputs/training_curves.png


In [2]:
import torch

# 如果 history 不在内存里，从文件加载
if 'history' not in dir() or not history['train_loss']:
    ckpt = torch.load('outputs/model_mlp_baseline.pt', map_location='cpu')
    history = ckpt['history']

print(f"\n=== Training Summary ===")
print(f"Initial train loss: {history['train_loss'][0]:.6f}")
print(f"Final train loss:   {history['train_loss'][-1]:.6f}")
print(f"Loss reduction:     {(history['train_loss'][0] - history['train_loss'][-1]) / history['train_loss'][0] * 100:.1f}%")

if history['test_loss']:
    print(f"\nInitial test loss:  {history['test_loss'][0]:.6f}")
    print(f"Final test loss:    {history['test_loss'][-1]:.6f}")


=== Training Summary ===
Initial train loss: 0.113078
Final train loss:   0.002963
Loss reduction:     97.4%

Initial test loss:  0.066700
Final test loss:    0.028925


## 6. 保存模型

In [10]:
# 保存模型
model_save_path = Path.cwd() / 'outputs' / 'model_mlp_baseline.pt'
model_save_path.parent.mkdir(exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
}, model_save_path)

print(f"Model saved to {model_save_path}")
print(f"Model size: {model_save_path.stat().st_size / 1e6:.1f} MB")

Model saved to /mnt/d/temp/shixi/shixi_project/Project_A/outputs/model_mlp_baseline.pt
Model size: 136.6 MB


## 7. 快速推理测试

In [ ]:
# 在测试集上做一些随机推理
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    images = batch['images'].to(device)
    actions_gt = batch['actions'].to(device)
    
    pred_actions = model(images)
    
    print("\n=== Inference Examples ===")
    for i in range(min(3, len(pred_actions))):
        print(f"\nSample {i}:")
        print(f"  GT action:   {actions_gt[i].cpu().numpy()}")
        print(f"  Pred action: {pred_actions[i].cpu().numpy()}")
        error = torch.abs(actions_gt[i] - pred_actions[i]).mean().item()
        print(f"  Mean error:  {error:.6f}")